# Train drilling advisory models with near_1, near_3 and near_5 horizons

Notebook расширяет текущий `train_drilling_advisory_lp`:

1. обучает отдельные пары моделей для горизонтов `near_1`, `near_3`, `near_5`;
2. для каждого горизонта считает ML-метрики:
   - `rotation_model_nearH`;
   - `speed_model_oracle_rotation_nearH`;
   - `speed_model_chained_nearH`;
3. для каждого горизонта запускает optimizer replay;
4. для каждого горизонта считает:
   - `model_based_predicted_uplift_pct`;
   - `predicted_regret_vs_operator_pct`;
   - `recommended_win_vs_operator_rate`;
   - `recommended_win_vs_operator_2pct_rate`;
   - `recommended_win_vs_operator_5pct_rate`;
   - `current_prediction_error_vs_operator_pct`;
   - `bias_adjusted_regret_vs_operator_pct`.

`near_5` остаётся основным горизонтом для симулятора и сохраняется в старые имена артефактов:

```text
rotation_model_near5.joblib
speed_model_near5.joblib
offline_recommendations_light_penalty.csv
```


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

try:
    import lightgbm as lgb
except Exception as exc:
    raise ImportError(
        "The final advisory notebook requires lightgbm. "
        "Install it with: pip install lightgbm"
    ) from exc

import joblib

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from config import (
        LABELED_DATA_PATH,
        DRILLING_ADVISORY_ARTIFACT_DIR,
        DRILLING_ADVISORY_REPORT_DIR,
        RANDOM_STATE,
        EPS,
        GRID_SIZE,
        MAX_DELTA_FRAC,
        FINAL_OPTIMIZER_MODE,
        CHANGE_PENALTY_WEIGHT,
        BOUNDARY_PENALTY_WEIGHT,
        BOUNDARY_START,
    )
except Exception:
    LABELED_DATA_PATH = PROJECT_ROOT / "notebooks" / "united_rock_energy_segment_quantile.csv"
    DRILLING_ADVISORY_ARTIFACT_DIR = PROJECT_ROOT / "notebooks" / "drilling_advisory_light_penalty_artifacts"
    DRILLING_ADVISORY_REPORT_DIR = PROJECT_ROOT / "notebooks" / "drilling_advisory_light_penalty_reports"
    RANDOM_STATE = 42
    EPS = 1e-9
    GRID_SIZE = 9
    MAX_DELTA_FRAC = 0.08
    FINAL_OPTIMIZER_MODE = "light_penalty"
    CHANGE_PENALTY_WEIGHT = 0.010
    BOUNDARY_PENALTY_WEIGHT = 0.020
    BOUNDARY_START = 0.85

DATA_PATH = Path(LABELED_DATA_PATH)
if not DATA_PATH.exists():
    alt = PROJECT_ROOT / "notebooks" / "united_rock_energy_segment_quantile.csv"
    if alt.exists():
        DATA_PATH = alt
    else:
        raise FileNotFoundError(f"Labeled data not found: {LABELED_DATA_PATH}")

ARTIFACT_DIR = Path(DRILLING_ADVISORY_ARTIFACT_DIR)
REPORT_DIR = Path(DRILLING_ADVISORY_REPORT_DIR)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_HORIZONS = [1, 3, 5]
DEFAULT_HORIZON = 5

EVAL_N = 2500
DIRECTION_DEADBAND_PCT = 0.5

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("TARGET_HORIZONS:", TARGET_HORIZONS)
print("DEFAULT_HORIZON:", DEFAULT_HORIZON)


PROJECT_ROOT: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling
DATA_PATH: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\united_rock_energy_segment_quantile.csv
ARTIFACT_DIR: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\drilling_advisory_light_penalty_artifacts
REPORT_DIR: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\drilling_advisory_light_penalty_reports
TARGET_HORIZONS: [1, 3, 5]
DEFAULT_HORIZON: 5


## 1. Load labeled data

In [2]:
df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")
df["processing_time"] = pd.to_datetime(df["processing_time"], errors="raise", format="mixed")
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)
df["rock_energy_type_final"] = df["rock_energy_type_final"].fillna("unknown").astype(str)

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "hardness_score_smooth",
    "rock_energy_type_final",
]

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df[required_cols].head())
display(
    df[["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]]
    .describe(percentiles=[.01, .05, .5, .95, .99])
)


Loaded: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\united_rock_energy_segment_quantile.csv
Shape: (415049, 87)


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755,NaN,unknown
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030,NaN,unknown
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060,NaN,unknown
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755,NaN,unknown
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515,NaN,unknown


,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth
count,415049.000000,415049.000000,415049.000000,415049.000000,3.826350e+05
mean,17473.667273,14134.146325,103.945315,0.013116,2.120217e-15
std,4682.982816,3243.523440,13.370361,0.006615,9.999975e-01
min,317.000000,784.000000,50.010000,0.001002,-6.036494e+00
1%,3713.000000,6271.000000,64.980000,0.002755,-2.698224e+00
5%,6645.000000,8246.000000,81.750000,0.005050,-1.737642e+00
50%,18861.000000,14637.000000,103.158000,0.012120,8.955322e-02
95%,22343.000000,18758.600000,138.474000,0.024240,1.660707e+00
99%,23626.000000,20881.000000,139.020000,0.030300,2.116441e+00
max,24872.000000,26318.000000,139.578000,0.038957,3.723870e+00


## 2. Feature engineering

In [3]:
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    out["total_pressure"] = out["pressure_axis"] + out["pressure_rotation"]
    out["pressure_balance"] = out["pressure_axis"] / (out["total_pressure"] + EPS)
    out["axis_over_rot_pressure"] = out["pressure_axis"] / (out["pressure_rotation"] + EPS)
    out["rot_pressure_over_axis"] = out["pressure_rotation"] / (out["pressure_axis"] + EPS)
    out["rotation_efficiency"] = out["rotation"] / (out["pressure_rotation"] + EPS)
    out["axis_x_rotation"] = out["pressure_axis"] * out["rotation"]
    out["rot_pressure_x_rotation"] = out["pressure_rotation"] * out["rotation"]
    out["energy_input_proxy"] = out["pressure_axis"] + out["pressure_rotation"] * out["rotation"]
    out["log_energy_input_proxy"] = np.log1p(out["energy_input_proxy"].clip(lower=0))

    out["dt"] = out.groupby("well_id")["processing_time"].diff().dt.total_seconds()
    out["dt"] = out["dt"].replace([np.inf, -np.inf], np.nan)
    out["dt"] = out["dt"].fillna(out["dt"].median())

    history_cols = [
        "pressure_axis",
        "pressure_rotation",
        "rotation",
        "speed",
        "hardness_score_smooth",
        "energy_input_proxy",
        "pressure_balance",
    ]

    for col in history_cols:
        grp = out.groupby("well_id")[col]

        for lag in [1, 3, 6, 12]:
            out[f"{col}_lag{lag}"] = grp.shift(lag)

        shifted = grp.shift(1)
        for w in [6, 12, 30]:
            min_p = max(2, w // 3)
            out[f"{col}_roll_mean_{w}"] = (
                shifted.groupby(out["well_id"])
                .rolling(w, min_periods=min_p)
                .mean()
                .reset_index(level=0, drop=True)
            )
            out[f"{col}_roll_std_{w}"] = (
                shifted.groupby(out["well_id"])
                .rolling(w, min_periods=min_p)
                .std()
                .reset_index(level=0, drop=True)
            )

    for col in ["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]:
        prev = out.groupby("well_id")[col].shift(1)
        out[f"{col}_diff1"] = out[col] - prev

    return out


df = add_features(df)
display(df.head())


,processing_time,depth_m,rotation,pressure_axis,pressure_rotation,well_id,speed,dt,total_pressure,pressure_balance,...,pressure_balance_lag3,pressure_balance_lag6,pressure_balance_lag12,pressure_balance_roll_mean_6,pressure_balance_roll_std_6,pressure_axis_diff1,pressure_rotation_diff1,rotation_diff1,speed_diff1,hardness_score_smooth_diff1
0,2025-08-24 10:09:50.980,0.0606,74.256,804,4551,19601,0.002755,5.135,5355,0.150140,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-08-24 10:10:00.260,0.0909,73.812,763,3782,19601,0.003030,9.280,4545,0.167877,...,NaN,NaN,NaN,NaN,NaN,-41.0,-769.0,-0.444,0.000275,NaN
2,2025-08-24 10:10:05.199,0.1212,73.512,879,4407,19601,0.006060,4.939,5286,0.166288,...,NaN,NaN,NaN,0.159008,0.012542,116.0,625.0,-0.300,0.003030,NaN
3,2025-08-24 10:10:14.610,0.1515,73.962,721,3705,19601,0.002755,9.411,4426,0.162901,...,0.150140,NaN,NaN,0.161435,0.009814,-158.0,-702.0,0.450,-0.003305,NaN
4,2025-08-24 10:10:34.197,0.1818,74.256,859,3883,19601,0.001515,19.587,4742,0.181147,...,0.167877,NaN,NaN,0.161802,0.008047,138.0,178.0,0.294,-0.001240,NaN


## 3. Targets for near_1, near_3 and near_5

In [4]:
def future_mean_by_group(data: pd.DataFrame, value_col: str, horizon: int) -> pd.Series:
    # Mean of t+1 ... t+horizon inside each well.
    # For horizon=1 this is just the next step value.
    return (
        data.groupby("well_id")[value_col]
        .transform(
            lambda s: (
                s.shift(-1)
                .rolling(horizon, min_periods=horizon)
                .mean()
                .shift(-(horizon - 1))
            )
        )
    )


target_cols_by_horizon = {}

for h in TARGET_HORIZONS:
    suffix = f"near{h}"

    target_rotation = f"target_rotation_{suffix}"
    target_speed = f"target_speed_{suffix}"
    target_pressure_axis = f"target_pressure_axis_{suffix}"
    target_pressure_rotation = f"target_pressure_rotation_{suffix}"

    df[target_rotation] = future_mean_by_group(df, "rotation", h)
    df[target_speed] = future_mean_by_group(df, "speed", h)

    # Operator's factual future controls. These are not ML targets; they are used
    # for offline sanity checks of recommendation direction.
    df[target_pressure_axis] = future_mean_by_group(df, "pressure_axis", h)
    df[target_pressure_rotation] = future_mean_by_group(df, "pressure_rotation", h)

    target_cols_by_horizon[h] = {
        "target_rotation": target_rotation,
        "target_speed": target_speed,
        "target_pressure_axis": target_pressure_axis,
        "target_pressure_rotation": target_pressure_rotation,
    }

display(
    df[
        [
            "rotation",
            "speed",
            "pressure_axis",
            "pressure_rotation",
            "target_rotation_near1",
            "target_speed_near1",
            "target_rotation_near3",
            "target_speed_near3",
            "target_rotation_near5",
            "target_speed_near5",
        ]
    ].describe(percentiles=[.01, .05, .5, .95, .99])
)


,rotation,speed,pressure_axis,pressure_rotation,target_rotation_near1,target_speed_near1,target_rotation_near3,target_speed_near3,target_rotation_near5,target_speed_near5
count,415049.000000,415049.000000,415049.000000,415049.000000,413343.000000,413343.000000,409931.000000,409931.000000,406519.000000,406519.000000
mean,103.945315,0.013116,17473.667273,14134.146325,103.988937,0.013109,104.016438,0.013111,104.047305,0.013107
std,13.370361,0.006615,4682.982816,3243.523440,13.267876,0.006597,10.706074,0.005624,10.173528,0.005406
min,50.010000,0.001002,317.000000,784.000000,50.010000,0.001002,50.638000,0.001175,50.868000,0.001409
1%,64.980000,0.002755,3713.000000,6271.000000,65.508000,0.002755,67.410000,0.003612,68.856432,0.003813
5%,81.750000,0.005050,6645.000000,8246.000000,82.110000,0.005050,84.244000,0.005681,84.986400,0.005656
50%,103.158000,0.012120,18861.000000,14637.000000,103.158000,0.012120,103.290000,0.012120,103.300800,0.012524
95%,138.474000,0.024240,22343.000000,18758.600000,138.474000,0.024240,126.580000,0.023567,123.786000,0.023230
99%,139.020000,0.030300,23626.000000,20881.000000,139.020000,0.030300,128.092000,0.028953,127.534584,0.028280
max,139.578000,0.038957,24872.000000,26318.000000,139.578000,0.038957,139.474000,0.037731,139.317600,0.036360


## 4. Feature set

In [5]:
# Агрессивно сокращённый набор признаков.
# Управляющие признаки pressure_axis / pressure_rotation оставлены обязательно,
# потому что optimizer именно ими управляет.
HARDNESS_FEATURE_COLUMNS = [
    "hardness_score_smooth",
]

base_numeric_features = [
    # current controls / state
    "pressure_axis", "pressure_rotation", "pressure_balance",
    "rotation", "speed", "dt",
    "rotation_efficiency", "axis_x_rotation",
    "energy_input_proxy",

    # stable energy state
    *HARDNESS_FEATURE_COLUMNS,

    # pressure history
    "pressure_axis_lag1", "pressure_axis_lag3", "pressure_axis_lag6",

    # rotation history
    "rotation_lag1", "rotation_lag3", "rotation_lag6",

    # speed history
    "speed_lag1", "speed_lag3",

    # stable hardness history
    "hardness_score_smooth_lag1", "hardness_score_smooth_lag3", "hardness_score_smooth_lag6",

    # rolling context
    "pressure_axis_roll_mean_12", "pressure_axis_roll_std_12",
    "pressure_rotation_roll_mean_12", "pressure_rotation_roll_std_12",
    "rotation_roll_mean_12", "rotation_roll_std_12",
    "speed_roll_mean_12", "speed_roll_std_12",
    "hardness_score_smooth_roll_mean_12",

    # short dynamics
    "pressure_axis_diff1", "pressure_rotation_diff1", "rotation_diff1",
    "speed_diff1",
]

categorical_features = ["rock_energy_type_final"]

speed_extra_features = ["candidate_target_rotation"]
speed_numeric_features = base_numeric_features + speed_extra_features

PRUNED_FEATURES_REMOVED = [
    "hardness_score",
    "hardness_score_smooth_12",
    "hardness_score_smooth_30",
    "pressure_axis_rel_diff1",
    "pressure_rotation_rel_diff1",
    "rotation_rel_diff1",
    "speed_rel_diff1",
    "hardness_score_smooth_rel_diff1",
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "log_energy_input_proxy",
    "rot_pressure_x_rotation",
    "total_pressure",
    "speed_lag6",
    "pressure_rotation_lag1",
    "pressure_rotation_lag3",
    "pressure_rotation_lag6",
    "hardness_score_smooth_roll_std_12",
    "hardness_score_smooth_diff1",
]

all_target_cols = []
for cols in target_cols_by_horizon.values():
    all_target_cols.extend(cols.values())

all_required = base_numeric_features + categorical_features + all_target_cols
missing = [c for c in all_required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# One common model_df and one common split for comparable near_1/3/5 metrics.
model_df = df.dropna(subset=all_required + ["well_id", "processing_time"]).copy()

print("Model df:", model_df.shape)
print("Numeric features:", len(base_numeric_features))
print("Categorical features:", categorical_features)
print("Hardness features:", HARDNESS_FEATURE_COLUMNS)
print("Removed features:", PRUNED_FEATURES_REMOVED)


Model df: (363869, 158)
Numeric features: 34
Categorical features: ['rock_energy_type_final']
Hardness features: ['hardness_score_smooth']
Removed features: ['hardness_score', 'hardness_score_smooth_12', 'hardness_score_smooth_30', 'pressure_axis_rel_diff1', 'pressure_rotation_rel_diff1', 'rotation_rel_diff1', 'speed_rel_diff1', 'hardness_score_smooth_rel_diff1', 'axis_over_rot_pressure', 'rot_pressure_over_axis', 'log_energy_input_proxy', 'rot_pressure_x_rotation', 'total_pressure', 'speed_lag6', 'pressure_rotation_lag1', 'pressure_rotation_lag3', 'pressure_rotation_lag6', 'hardness_score_smooth_roll_std_12', 'hardness_score_smooth_diff1']


## 5. Train/test split

In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["well_id"]))

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train:", train_df.shape, "wells:", train_df["well_id"].nunique())
print("Test:", test_df.shape, "wells:", test_df["well_id"].nunique())
print("Test wells:", list(test_df["well_id"].unique())[:10])


Train: (270511, 158) wells: 1279
Test: (93358, 158) wells: 427
Test wells: [np.int64(19677), np.int64(19720), np.int64(19775), np.int64(19780), np.int64(19789), np.int64(19905), np.int64(19909), np.int64(19935), np.int64(19958), np.int64(19977)]


## 6. Model helpers

In [7]:
def make_regressor(seed_offset: int = 0):
    return lgb.LGBMRegressor(
        n_estimators=650,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE + seed_offset,
        objective="regression",
        verbosity=-1,
    )


def make_preprocessor(numeric_features, categorical_features):
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    return ColumnTransformer(
        transformers=[
            ("cat", encoder, categorical_features),
            ("num", "passthrough", numeric_features),
        ],
        remainder="drop",
    )


def regression_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(root_mean_squared_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def train_models_for_horizon(horizon: int):
    targets = target_cols_by_horizon[horizon]
    target_rotation = targets["target_rotation"]
    target_speed = targets["target_speed"]

    rotation_model = Pipeline(steps=[
        ("preprocess", make_preprocessor(base_numeric_features, categorical_features)),
        ("model", make_regressor(seed_offset=horizon)),
    ])

    rotation_model.fit(
        train_df[base_numeric_features + categorical_features],
        train_df[target_rotation],
    )

    pred_target_rotation = rotation_model.predict(
        test_df[base_numeric_features + categorical_features]
    )

    rotation_metrics = regression_metrics(
        test_df[target_rotation],
        pred_target_rotation,
    )

    train_speed_df = train_df.copy()
    train_speed_df["candidate_target_rotation"] = train_speed_df[target_rotation]

    test_speed_oracle_df = test_df.copy()
    test_speed_oracle_df["candidate_target_rotation"] = test_speed_oracle_df[target_rotation]

    test_speed_chained_df = test_df.copy()
    test_speed_chained_df["candidate_target_rotation"] = pred_target_rotation

    speed_model = Pipeline(steps=[
        ("preprocess", make_preprocessor(speed_numeric_features, categorical_features)),
        ("model", make_regressor(seed_offset=100 + horizon)),
    ])

    speed_model.fit(
        train_speed_df[speed_numeric_features + categorical_features],
        train_speed_df[target_speed],
    )

    pred_speed_oracle = speed_model.predict(
        test_speed_oracle_df[speed_numeric_features + categorical_features]
    )

    pred_speed_chained = speed_model.predict(
        test_speed_chained_df[speed_numeric_features + categorical_features]
    )

    speed_metrics_oracle = regression_metrics(
        test_df[target_speed],
        pred_speed_oracle,
    )

    speed_metrics_chained = regression_metrics(
        test_df[target_speed],
        pred_speed_chained,
    )

    return {
        "horizon": horizon,
        "rotation_model": rotation_model,
        "speed_model": speed_model,
        "pred_target_rotation": pred_target_rotation,
        "pred_speed_oracle": pred_speed_oracle,
        "pred_speed_chained": pred_speed_chained,
        "rotation_metrics": rotation_metrics,
        "speed_metrics_oracle": speed_metrics_oracle,
        "speed_metrics_chained": speed_metrics_chained,
    }


## 7. Train near_1, near_3 and near_5 models

In [8]:
models_by_horizon = {}
metrics_rows = []

for h in TARGET_HORIZONS:
    print("=" * 100)
    print(f"Training horizon near_{h}")

    result = train_models_for_horizon(h)
    models_by_horizon[h] = result

    metrics_rows.extend([
        {"horizon": h, "model": f"rotation_model_near{h}", **result["rotation_metrics"]},
        {"horizon": h, "model": f"speed_model_oracle_rotation_near{h}", **result["speed_metrics_oracle"]},
        {"horizon": h, "model": f"speed_model_chained_near{h}", **result["speed_metrics_chained"]},
    ])

    print(f"Rotation model near_{h}:")
    print(result["rotation_metrics"])

    print(f"Speed model near_{h} with actual target rotation:")
    print(result["speed_metrics_oracle"])

    print(f"Speed model near_{h} chained:")
    print(result["speed_metrics_chained"])

metrics_summary = pd.DataFrame(metrics_rows)
display(metrics_summary)

metrics_summary.to_csv(ARTIFACT_DIR / "training_metrics_by_horizon.csv", index=False)
metrics_summary.to_csv(REPORT_DIR / "training_metrics_by_horizon.csv", index=False)

# Backward-compatible single-horizon file for the default near_5 model.
default_result = models_by_horizon[DEFAULT_HORIZON]
training_metrics_default = pd.DataFrame([
    {"model": f"rotation_model_near{DEFAULT_HORIZON}", **default_result["rotation_metrics"]},
    {"model": f"speed_model_oracle_rotation_near{DEFAULT_HORIZON}", **default_result["speed_metrics_oracle"]},
    {"model": f"speed_model_chained_near{DEFAULT_HORIZON}", **default_result["speed_metrics_chained"]},
])
training_metrics_default.to_csv(ARTIFACT_DIR / "training_metrics.csv", index=False)
training_metrics_default.to_csv(REPORT_DIR / "training_metrics.csv", index=False)
display(training_metrics_default)


Training horizon near_1
Rotation model near_1:
{'MAE': 3.897782505312641, 'RMSE': 7.870523063252685, 'R2': 0.5034706006405628}
Speed model near_1 with actual target rotation:
{'MAE': 0.003207890495889067, 'RMSE': 0.004148977938644388, 'R2': 0.5834828795269961}
Speed model near_1 chained:
{'MAE': 0.0032570008405644587, 'RMSE': 0.004216920147903098, 'R2': 0.5697297086012874}
Training horizon near_3
Rotation model near_3:
{'MAE': 2.0386841877105066, 'RMSE': 4.378190535278904, 'R2': 0.7550020200980115}
Speed model near_3 with actual target rotation:
{'MAE': 0.0020635215776645868, 'RMSE': 0.002782149071999733, 'R2': 0.742448731681968}
Speed model near_3 chained:
{'MAE': 0.002159269873687146, 'RMSE': 0.002916663271323488, 'R2': 0.7169419663157978}
Training horizon near_5
Rotation model near_5:
{'MAE': 1.7341025882898387, 'RMSE': 3.5775783532199643, 'R2': 0.8185756306426544}
Speed model near_5 with actual target rotation:
{'MAE': 0.0019358429796978108, 'RMSE': 0.0026216917252666594, 'R2': 0.7

,horizon,model,MAE,RMSE,R2
0,1,rotation_model_near1,3.897783,7.870523,0.503471
1,1,speed_model_oracle_rotation_near1,0.003208,0.004149,0.583483
2,1,speed_model_chained_near1,0.003257,0.004217,0.569730
3,3,rotation_model_near3,2.038684,4.378191,0.755002
4,3,speed_model_oracle_rotation_near3,0.002064,0.002782,0.742449
5,3,speed_model_chained_near3,0.002159,0.002917,0.716942
6,5,rotation_model_near5,1.734103,3.577578,0.818576
7,5,speed_model_oracle_rotation_near5,0.001936,0.002622,0.754808
8,5,speed_model_chained_near5,0.002028,0.002732,0.733680


,model,MAE,RMSE,R2
0,rotation_model_near5,1.734103,3.577578,0.818576
1,speed_model_oracle_rotation_near5,0.001936,0.002622,0.754808
2,speed_model_chained_near5,0.002028,0.002732,0.733680


## 8. Baselines and candidate bounds

In [9]:
baseline_rows = []

for h in TARGET_HORIZONS:
    target_speed = target_cols_by_horizon[h]["target_speed"]
    chained_metrics = models_by_horizon[h]["speed_metrics_chained"]
    oracle_metrics = models_by_horizon[h]["speed_metrics_oracle"]

    baseline_current = regression_metrics(test_df[target_speed], test_df["speed"])
    baseline_roll12 = regression_metrics(
        test_df[target_speed],
        test_df["speed_roll_mean_12"].fillna(test_df["speed"]),
    )

    baseline_rows.extend([
        {"horizon": h, "model": "current_speed", **baseline_current},
        {"horizon": h, "model": "speed_roll_mean_12", **baseline_roll12},
        {"horizon": h, "model": f"rotation_to_speed_chained_near{h}", **chained_metrics},
        {"horizon": h, "model": f"speed_oracle_rotation_near{h}", **oracle_metrics},
    ])

baseline_compare = pd.DataFrame(baseline_rows).sort_values(["horizon", "MAE"])
display(baseline_compare)

surface_ranges = {}

for et, part in train_df.groupby("rock_energy_type_final"):
    if len(part) < 100:
        continue

    surface_ranges[et] = {
        "pressure_axis_q05": float(part["pressure_axis"].quantile(0.05)),
        "pressure_axis_q95": float(part["pressure_axis"].quantile(0.95)),
        "pressure_rotation_q05": float(part["pressure_rotation"].quantile(0.05)),
        "pressure_rotation_q95": float(part["pressure_rotation"].quantile(0.95)),
        "rotation_median": float(part["rotation"].median()),
        "speed_median": float(part["speed"].median()),
        "hardness_median": float(part["hardness_score_smooth"].median()),
        "rows": int(len(part)),
    }

surface_ranges_df = pd.DataFrame(surface_ranges).T
display(surface_ranges_df)

baseline_compare.to_csv(ARTIFACT_DIR / "baseline_compare_by_horizon.csv", index=False)
baseline_compare.to_csv(REPORT_DIR / "baseline_compare_by_horizon.csv", index=False)


,horizon,model,MAE,RMSE,R2
3,1,speed_oracle_rotation_near1,0.003208,0.004149,0.583483
2,1,rotation_to_speed_chained_near1,0.003257,0.004217,0.569730
1,1,speed_roll_mean_12,0.003662,0.004743,0.455662
0,1,current_speed,0.004577,0.006035,0.118706
7,3,speed_oracle_rotation_near3,0.002064,0.002782,0.742449
6,3,rotation_to_speed_chained_near3,0.002159,0.002917,0.716942
5,3,speed_roll_mean_12,0.002592,0.003535,0.584094
4,3,current_speed,0.003562,0.004607,0.293642
11,5,speed_oracle_rotation_near5,0.001936,0.002622,0.754808
10,5,rotation_to_speed_chained_near5,0.002028,0.002732,0.733680


,pressure_axis_q05,pressure_axis_q95,pressure_rotation_q05,pressure_rotation_q95,rotation_median,speed_median,hardness_median,rows
hard_high_energy,13524.0,22887.0,9895.0,18831.65,103.458,0.00606,1.110124,67028.0
medium_high_energy,14038.0,22680.0,10459.0,18877.00,102.864,0.01212,0.311631,69031.0
medium_low_energy,12543.0,22348.0,9752.0,19053.00,102.966,0.01212,-0.213567,67855.0
soft_low_energy,7524.0,21374.0,8313.0,18575.00,103.410,0.01818,-1.071703,66597.0


## 9. Optimizer helpers

In [10]:
def drop_duplicate_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.loc[:, ~frame.columns.duplicated()].copy()


def scalar_from_row(row: pd.Series, col: str) -> float:
    value = row[col]
    if isinstance(value, pd.Series):
        value = value.iloc[0]
    return float(value)


def recompute_candidate_features(grid: pd.DataFrame) -> pd.DataFrame:
    grid = grid.copy()
    grid["total_pressure"] = grid["pressure_axis"] + grid["pressure_rotation"]
    grid["pressure_balance"] = grid["pressure_axis"] / (grid["total_pressure"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["axis_x_rotation"] = grid["pressure_axis"] * grid["rotation"]
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]
    return grid


def build_candidate_grid(row: pd.Series, grid_size: int = GRID_SIZE, max_delta_frac: float = MAX_DELTA_FRAC):
    et = row["rock_energy_type_final"]

    if et in surface_ranges:
        r = surface_ranges[et]
        p_ax_low = r["pressure_axis_q05"]
        p_ax_high = r["pressure_axis_q95"]
        p_rot_low = r["pressure_rotation_q05"]
        p_rot_high = r["pressure_rotation_q95"]
    else:
        p_ax_low = train_df["pressure_axis"].quantile(0.05)
        p_ax_high = train_df["pressure_axis"].quantile(0.95)
        p_rot_low = train_df["pressure_rotation"].quantile(0.05)
        p_rot_high = train_df["pressure_rotation"].quantile(0.95)

    cur_ax = float(row["pressure_axis"])
    cur_rotp = float(row["pressure_rotation"])

    local_ax_low = cur_ax * (1.0 - max_delta_frac)
    local_ax_high = cur_ax * (1.0 + max_delta_frac)
    local_rotp_low = cur_rotp * (1.0 - max_delta_frac)
    local_rotp_high = cur_rotp * (1.0 + max_delta_frac)

    p_ax_min = max(p_ax_low, local_ax_low)
    p_ax_max = min(p_ax_high, local_ax_high)
    p_rot_min = max(p_rot_low, local_rotp_low)
    p_rot_max = min(p_rot_high, local_rotp_high)

    if p_ax_min >= p_ax_max:
        p_ax_min, p_ax_max = local_ax_low, local_ax_high

    if p_rot_min >= p_rot_max:
        p_rot_min, p_rot_max = local_rotp_low, local_rotp_high

    p_ax_grid = np.linspace(p_ax_min, p_ax_max, grid_size)
    p_rot_grid = np.linspace(p_rot_min, p_rot_max, grid_size)
    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

    grid = pd.DataFrame({
        "pressure_axis": PA.ravel(),
        "pressure_rotation": PR.ravel(),
    })

    recomputed = {
        "pressure_axis", "pressure_rotation", "pressure_balance",
        "rotation_efficiency", "axis_x_rotation", "energy_input_proxy",
    }

    for col in base_numeric_features:
        if col not in recomputed:
            grid[col] = row[col]

    for col in categorical_features:
        grid[col] = row[col]

    grid = recompute_candidate_features(grid)

    grid["delta_pressure_axis_frac"] = grid["pressure_axis"] / (cur_ax + EPS) - 1.0
    grid["delta_pressure_rotation_frac"] = grid["pressure_rotation"] / (cur_rotp + EPS) - 1.0

    return drop_duplicate_columns(grid), PA, PR


def score_candidates(grid: pd.DataFrame) -> pd.DataFrame:
    grid = grid.copy()

    change_penalty = CHANGE_PENALTY_WEIGHT * (
        grid["delta_pressure_axis_frac"].abs()
        + grid["delta_pressure_rotation_frac"].abs()
    )

    max_abs_delta_frac = np.maximum(
        grid["delta_pressure_axis_frac"].abs(),
        grid["delta_pressure_rotation_frac"].abs(),
    )
    normalized_boundary = max_abs_delta_frac / (MAX_DELTA_FRAC + EPS)
    boundary_excess = np.maximum(0.0, normalized_boundary - BOUNDARY_START)
    boundary_penalty = BOUNDARY_PENALTY_WEIGHT * boundary_excess

    grid["change_penalty"] = change_penalty
    grid["boundary_penalty"] = boundary_penalty
    grid["optimizer_score"] = (
        grid["predicted_target_speed"]
        - grid["change_penalty"]
        - grid["boundary_penalty"]
    )

    return grid


def predict_current_target_speed(row: pd.Series, horizon: int) -> float:
    rotation_model = models_by_horizon[horizon]["rotation_model"]
    speed_model = models_by_horizon[horizon]["speed_model"]

    current_grid = pd.DataFrame([row[base_numeric_features + categorical_features].to_dict()])
    current_grid = drop_duplicate_columns(current_grid)

    current_grid["candidate_target_rotation"] = rotation_model.predict(
        current_grid[base_numeric_features + categorical_features]
    )

    return float(speed_model.predict(current_grid[speed_numeric_features + categorical_features])[0])


def recommend_for_row(row: pd.Series, horizon: int, grid_size: int = GRID_SIZE) -> dict:
    rotation_model = models_by_horizon[horizon]["rotation_model"]
    speed_model = models_by_horizon[horizon]["speed_model"]

    grid, _, _ = build_candidate_grid(row, grid_size=grid_size)
    grid = drop_duplicate_columns(grid)

    if "candidate_target_rotation" in grid.columns:
        grid = grid.drop(columns=["candidate_target_rotation"])
    if "predicted_target_speed" in grid.columns:
        grid = grid.drop(columns=["predicted_target_speed"])

    grid["candidate_target_rotation"] = rotation_model.predict(
        grid[base_numeric_features + categorical_features]
    )
    grid["predicted_target_speed"] = speed_model.predict(
        grid[speed_numeric_features + categorical_features]
    )

    grid = score_candidates(grid)
    best = grid.loc[grid["optimizer_score"].idxmax()]

    current_predicted_target_speed = predict_current_target_speed(row, horizon)
    best_predicted_target_speed = scalar_from_row(best, "predicted_target_speed")

    predicted_uplift_pct = 100.0 * (
        best_predicted_target_speed / (current_predicted_target_speed + EPS) - 1.0
    )
    score_uplift_pct = 100.0 * (
        scalar_from_row(best, "optimizer_score") / (current_predicted_target_speed + EPS) - 1.0
    )

    return {
        "recommended_pressure_axis": scalar_from_row(best, "pressure_axis"),
        "recommended_pressure_rotation": scalar_from_row(best, "pressure_rotation"),
        "candidate_target_rotation": scalar_from_row(best, "candidate_target_rotation"),
        "predicted_target_speed": best_predicted_target_speed,
        "optimizer_score": scalar_from_row(best, "optimizer_score"),
        "current_predicted_target_speed": current_predicted_target_speed,
        "predicted_uplift_pct": predicted_uplift_pct,
        "score_uplift_pct": score_uplift_pct,
        "delta_pressure_axis_pct": 100.0 * scalar_from_row(best, "delta_pressure_axis_frac"),
        "delta_pressure_rotation_pct": 100.0 * scalar_from_row(best, "delta_pressure_rotation_frac"),
        "change_penalty": scalar_from_row(best, "change_penalty"),
        "boundary_penalty": scalar_from_row(best, "boundary_penalty"),
    }


def signed_direction(delta_pct: float, deadband_pct: float = DIRECTION_DEADBAND_PCT) -> int:
    if delta_pct > deadband_pct:
        return 1
    if delta_pct < -deadband_pct:
        return -1
    return 0


def safe_mean(series: pd.Series) -> float:
    if len(series) == 0:
        return np.nan
    return float(series.mean())


## 10. Fast optimizer replay and operator comparison for each horizon

In [11]:
# Fast optimizer replay:
# 1) build all candidate grids once for eval_points;
# 2) batch-predict all candidates for each horizon;
# 3) choose best candidate by row_id;
# 4) compute operator-comparison metrics vectorized.

eval_n = min(EVAL_N, len(test_df))
eval_points = test_df.sample(eval_n, random_state=RANDOM_STATE).copy().reset_index(drop=True)
eval_points["row_id"] = np.arange(len(eval_points))

print("Eval rows:", len(eval_points))
print("Candidate grid size per row:", GRID_SIZE * GRID_SIZE)

# Build candidate grids once. This is horizon-independent.
candidate_frames = []
for _, row in eval_points.iterrows():
    grid, _, _ = build_candidate_grid(row, grid_size=GRID_SIZE)
    grid = drop_duplicate_columns(grid)
    grid["row_id"] = int(row["row_id"])
    candidate_frames.append(grid)

all_candidates_base = pd.concat(candidate_frames, ignore_index=True)
all_candidates_base = drop_duplicate_columns(all_candidates_base)

print("All candidates:", all_candidates_base.shape)

all_recommendations = []
optimizer_summary_rows = []
by_energy_frames = []
operator_comparison_frames = []
win_rate_frames = []

base_feature_cols = base_numeric_features + categorical_features
speed_feature_cols = speed_numeric_features + categorical_features

for h in TARGET_HORIZONS:
    print("=" * 100)
    print(f"Fast optimizer replay for near_{h}")

    targets = target_cols_by_horizon[h]
    target_speed = targets["target_speed"]
    target_pressure_axis = targets["target_pressure_axis"]
    target_pressure_rotation = targets["target_pressure_rotation"]

    rotation_model = models_by_horizon[h]["rotation_model"]
    speed_model = models_by_horizon[h]["speed_model"]

    # Vectorized current-mode prediction for eval points.
    current_df = eval_points.copy()
    current_df = drop_duplicate_columns(current_df)

    current_df["candidate_target_rotation"] = rotation_model.predict(
        current_df[base_feature_cols]
    )
    current_df["current_predicted_target_speed"] = speed_model.predict(
        current_df[speed_feature_cols]
    )

    # Vectorized candidate prediction.
    cand = all_candidates_base.copy()
    cand = drop_duplicate_columns(cand)

    if "candidate_target_rotation" in cand.columns:
        cand = cand.drop(columns=["candidate_target_rotation"])
    if "predicted_target_speed" in cand.columns:
        cand = cand.drop(columns=["predicted_target_speed"])

    cand["candidate_target_rotation"] = rotation_model.predict(cand[base_feature_cols])
    cand["predicted_target_speed"] = speed_model.predict(cand[speed_feature_cols])

    cand = score_candidates(cand)

    # Select best candidate for each eval row.
    best_idx = cand.groupby("row_id", sort=False)["optimizer_score"].idxmax()
    best = cand.loc[best_idx].copy().sort_values("row_id").reset_index(drop=True)

    # Join current row context.
    context_cols = [
        "row_id",
        "well_id",
        "processing_time",
        "rock_energy_type_final",
        "pressure_axis",
        "pressure_rotation",
        "rotation",
        "speed",
        target_speed,
        target_pressure_axis,
        target_pressure_rotation,
        "current_predicted_target_speed",
    ]

    rec_df_h = current_df[context_cols].merge(
        best[
            [
                "row_id",
                "pressure_axis",
                "pressure_rotation",
                "candidate_target_rotation",
                "predicted_target_speed",
                "optimizer_score",
                "delta_pressure_axis_frac",
                "delta_pressure_rotation_frac",
                "change_penalty",
                "boundary_penalty",
            ]
        ],
        on="row_id",
        how="left",
        suffixes=("_current", "_recommended"),
    )

    # Normalize names.
    rec_df_h = rec_df_h.rename(columns={
        "pressure_axis_current": "current_pressure_axis",
        "pressure_rotation_current": "current_pressure_rotation",
        "rotation": "current_rotation",
        "speed": "current_speed",
        target_speed: "target_speed_actual",
        target_pressure_axis: "operator_future_pressure_axis",
        target_pressure_rotation: "operator_future_pressure_rotation",
        "pressure_axis_recommended": "recommended_pressure_axis",
        "pressure_rotation_recommended": "recommended_pressure_rotation",
        "candidate_target_rotation": "recommended_candidate_target_rotation",
        "predicted_target_speed": "recommended_predicted_target_speed",
    })

    rec_df_h["horizon"] = h
    rec_df_h["horizon_name"] = f"near_{h}"

    # Backward-compatible aliases.
    rec_df_h["actual_operator_target_speed"] = rec_df_h["target_speed_actual"]
    rec_df_h["actual_operator_mean_speed_t_plus_1_to_h"] = rec_df_h["target_speed_actual"]

    # Main model-based uplift:
    # recommended prediction vs current-mode prediction.
    rec_df_h["model_based_predicted_uplift_pct"] = 100.0 * (
        rec_df_h["recommended_predicted_target_speed"]
        / (rec_df_h["current_predicted_target_speed"] + EPS)
        - 1.0
    )
    rec_df_h["predicted_uplift_pct"] = rec_df_h["model_based_predicted_uplift_pct"]

    rec_df_h["score_uplift_pct"] = 100.0 * (
        rec_df_h["optimizer_score"]
        / (rec_df_h["current_predicted_target_speed"] + EPS)
        - 1.0
    )

    # Operator-comparison metrics:
    # recommended prediction vs actual operator outcome on t+1...t+H.
    rec_df_h["predicted_regret_vs_operator_speed"] = (
        rec_df_h["recommended_predicted_target_speed"]
        - rec_df_h["target_speed_actual"]
    )
    rec_df_h["predicted_regret_vs_operator_pct"] = 100.0 * (
        rec_df_h["recommended_predicted_target_speed"]
        / (rec_df_h["target_speed_actual"] + EPS)
        - 1.0
    )

    # Current prediction bias vs actual operator outcome.
    rec_df_h["current_prediction_error_vs_operator_speed"] = (
        rec_df_h["current_predicted_target_speed"]
        - rec_df_h["target_speed_actual"]
    )
    rec_df_h["current_prediction_error_vs_operator_pct"] = 100.0 * (
        rec_df_h["current_predicted_target_speed"]
        / (rec_df_h["target_speed_actual"] + EPS)
        - 1.0
    )

    rec_df_h["bias_adjusted_regret_vs_operator_pct"] = (
        rec_df_h["predicted_regret_vs_operator_pct"]
        - rec_df_h["current_prediction_error_vs_operator_pct"]
    )

    rec_df_h["recommended_win_vs_operator"] = (
        rec_df_h["recommended_predicted_target_speed"]
        > rec_df_h["target_speed_actual"]
    )
    rec_df_h["recommended_win_vs_operator_2pct"] = (
        rec_df_h["recommended_predicted_target_speed"]
        > rec_df_h["target_speed_actual"] * 1.02
    )
    rec_df_h["recommended_win_vs_operator_5pct"] = (
        rec_df_h["recommended_predicted_target_speed"]
        > rec_df_h["target_speed_actual"] * 1.05
    )
    rec_df_h["current_pred_above_operator"] = (
        rec_df_h["current_predicted_target_speed"]
        > rec_df_h["target_speed_actual"]
    )

    # Movement diagnostics.
    rec_df_h["delta_pressure_axis_pct"] = 100.0 * rec_df_h["delta_pressure_axis_frac"]
    rec_df_h["delta_pressure_rotation_pct"] = 100.0 * rec_df_h["delta_pressure_rotation_frac"]

    rec_df_h["operator_delta_pressure_axis_pct"] = 100.0 * (
        rec_df_h["operator_future_pressure_axis"]
        / (rec_df_h["current_pressure_axis"] + EPS)
        - 1.0
    )
    rec_df_h["operator_delta_pressure_rotation_pct"] = 100.0 * (
        rec_df_h["operator_future_pressure_rotation"]
        / (rec_df_h["current_pressure_rotation"] + EPS)
        - 1.0
    )

    rec_df_h["rec_axis_direction"] = rec_df_h["delta_pressure_axis_pct"].apply(signed_direction)
    rec_df_h["rec_rot_direction"] = rec_df_h["delta_pressure_rotation_pct"].apply(signed_direction)
    rec_df_h["operator_axis_direction"] = rec_df_h["operator_delta_pressure_axis_pct"].apply(signed_direction)
    rec_df_h["operator_rot_direction"] = rec_df_h["operator_delta_pressure_rotation_pct"].apply(signed_direction)

    boundary_tol = 100.0 * MAX_DELTA_FRAC * 0.999
    rec_df_h["axis_near_boundary"] = rec_df_h["delta_pressure_axis_pct"].abs() >= boundary_tol
    rec_df_h["rot_near_boundary"] = rec_df_h["delta_pressure_rotation_pct"].abs() >= boundary_tol
    rec_df_h["any_boundary"] = rec_df_h["axis_near_boundary"] | rec_df_h["rot_near_boundary"]

    rec_df_h["recommended_pred_above_actual"] = rec_df_h["recommended_win_vs_operator"]
    rec_df_h["current_pred_above_actual"] = rec_df_h["current_pred_above_operator"]

    rec_df_h["same_direction_axis"] = (
        rec_df_h["rec_axis_direction"] == rec_df_h["operator_axis_direction"]
    )
    rec_df_h["same_direction_rotation"] = (
        rec_df_h["rec_rot_direction"] == rec_df_h["operator_rot_direction"]
    )
    rec_df_h["same_direction_both"] = (
        rec_df_h["same_direction_axis"] & rec_df_h["same_direction_rotation"]
    )

    rec_df_h["direction_comparable_axis"] = (
        (rec_df_h["rec_axis_direction"] != 0) | (rec_df_h["operator_axis_direction"] != 0)
    )
    rec_df_h["direction_comparable_rotation"] = (
        (rec_df_h["rec_rot_direction"] != 0) | (rec_df_h["operator_rot_direction"] != 0)
    )
    rec_df_h["direction_comparable_both"] = (
        rec_df_h["direction_comparable_axis"] | rec_df_h["direction_comparable_rotation"]
    )

    summary_h = {
        "horizon": h,
        "horizon_name": f"near_{h}",
        "optimizer_mode": FINAL_OPTIMIZER_MODE,
        "rows": len(rec_df_h),

        "median_model_based_predicted_uplift_pct": rec_df_h["model_based_predicted_uplift_pct"].median(),
        "mean_model_based_predicted_uplift_pct": rec_df_h["model_based_predicted_uplift_pct"].mean(),
        "p05_model_based_predicted_uplift_pct": rec_df_h["model_based_predicted_uplift_pct"].quantile(0.05),
        "p95_model_based_predicted_uplift_pct": rec_df_h["model_based_predicted_uplift_pct"].quantile(0.95),

        "median_predicted_regret_vs_operator_pct": rec_df_h["predicted_regret_vs_operator_pct"].median(),
        "mean_predicted_regret_vs_operator_pct": rec_df_h["predicted_regret_vs_operator_pct"].mean(),
        "p05_predicted_regret_vs_operator_pct": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.05),
        "p25_predicted_regret_vs_operator_pct": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.25),
        "p75_predicted_regret_vs_operator_pct": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.75),
        "p95_predicted_regret_vs_operator_pct": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.95),

        "recommended_win_vs_operator_rate": rec_df_h["recommended_win_vs_operator"].mean(),
        "recommended_win_vs_operator_2pct_rate": rec_df_h["recommended_win_vs_operator_2pct"].mean(),
        "recommended_win_vs_operator_5pct_rate": rec_df_h["recommended_win_vs_operator_5pct"].mean(),

        # Backward-compatible names.
        "median_uplift_pct": rec_df_h["model_based_predicted_uplift_pct"].median(),
        "mean_uplift_pct": rec_df_h["model_based_predicted_uplift_pct"].mean(),
        "median_uplift_vs_actual_pct": rec_df_h["predicted_regret_vs_operator_pct"].median(),
        "mean_uplift_vs_actual_pct": rec_df_h["predicted_regret_vs_operator_pct"].mean(),
        "share_recommended_pred_above_actual": rec_df_h["recommended_win_vs_operator"].mean(),

        "median_current_prediction_error_vs_operator_pct": rec_df_h["current_prediction_error_vs_operator_pct"].median(),
        "mean_current_prediction_error_vs_operator_pct": rec_df_h["current_prediction_error_vs_operator_pct"].mean(),
        "share_current_pred_above_operator": rec_df_h["current_pred_above_operator"].mean(),
        "median_bias_adjusted_regret_vs_operator_pct": rec_df_h["bias_adjusted_regret_vs_operator_pct"].median(),
        "mean_bias_adjusted_regret_vs_operator_pct": rec_df_h["bias_adjusted_regret_vs_operator_pct"].mean(),

        "same_direction_axis_rate": safe_mean(
            rec_df_h.loc[rec_df_h["direction_comparable_axis"], "same_direction_axis"]
        ),
        "same_direction_rotation_rate": safe_mean(
            rec_df_h.loc[rec_df_h["direction_comparable_rotation"], "same_direction_rotation"]
        ),
        "same_direction_both_rate": safe_mean(
            rec_df_h.loc[rec_df_h["direction_comparable_both"], "same_direction_both"]
        ),

        "median_delta_axis_pct": rec_df_h["delta_pressure_axis_pct"].median(),
        "median_delta_rot_pct": rec_df_h["delta_pressure_rotation_pct"].median(),
        "median_abs_delta_axis_pct": rec_df_h["delta_pressure_axis_pct"].abs().median(),
        "median_abs_delta_rot_pct": rec_df_h["delta_pressure_rotation_pct"].abs().median(),
        "median_operator_delta_axis_pct": rec_df_h["operator_delta_pressure_axis_pct"].median(),
        "median_operator_delta_rot_pct": rec_df_h["operator_delta_pressure_rotation_pct"].median(),
        "axis_boundary_ratio": rec_df_h["axis_near_boundary"].mean(),
        "rot_boundary_ratio": rec_df_h["rot_near_boundary"].mean(),
        "any_boundary_ratio": rec_df_h["any_boundary"].mean(),
    }

    optimizer_summary_rows.append(summary_h)

    by_energy_h = (
        rec_df_h
        .groupby("rock_energy_type_final")
        .agg(
            rows=("model_based_predicted_uplift_pct", "size"),
            median_model_based_predicted_uplift_pct=("model_based_predicted_uplift_pct", "median"),
            mean_model_based_predicted_uplift_pct=("model_based_predicted_uplift_pct", "mean"),
            median_predicted_regret_vs_operator_pct=("predicted_regret_vs_operator_pct", "median"),
            mean_predicted_regret_vs_operator_pct=("predicted_regret_vs_operator_pct", "mean"),
            recommended_win_vs_operator_rate=("recommended_win_vs_operator", "mean"),
            recommended_win_vs_operator_2pct_rate=("recommended_win_vs_operator_2pct", "mean"),
            recommended_win_vs_operator_5pct_rate=("recommended_win_vs_operator_5pct", "mean"),
            median_current_prediction_error_vs_operator_pct=("current_prediction_error_vs_operator_pct", "median"),
            median_bias_adjusted_regret_vs_operator_pct=("bias_adjusted_regret_vs_operator_pct", "median"),
            median_abs_delta_axis_pct=("delta_pressure_axis_pct", lambda s: s.abs().median()),
            median_abs_delta_rot_pct=("delta_pressure_rotation_pct", lambda s: s.abs().median()),
            boundary_ratio=("any_boundary", "mean"),
        )
        .reset_index()
    )
    by_energy_h.insert(0, "horizon", h)
    by_energy_h.insert(1, "horizon_name", f"near_{h}")
    by_energy_frames.append(by_energy_h)

    operator_comparison_h = pd.DataFrame([
        {
            "horizon": h,
            "horizon_name": f"near_{h}",
            "metric": "model_based_predicted_uplift_pct = recommended_pred / current_pred - 1",
            "median": rec_df_h["model_based_predicted_uplift_pct"].median(),
            "mean": rec_df_h["model_based_predicted_uplift_pct"].mean(),
            "p05": rec_df_h["model_based_predicted_uplift_pct"].quantile(0.05),
            "p25": rec_df_h["model_based_predicted_uplift_pct"].quantile(0.25),
            "p75": rec_df_h["model_based_predicted_uplift_pct"].quantile(0.75),
            "p95": rec_df_h["model_based_predicted_uplift_pct"].quantile(0.95),
        },
        {
            "horizon": h,
            "horizon_name": f"near_{h}",
            "metric": "predicted_regret_vs_operator_pct = recommended_pred / actual_operator_mean_speed_t+1_to_h - 1",
            "median": rec_df_h["predicted_regret_vs_operator_pct"].median(),
            "mean": rec_df_h["predicted_regret_vs_operator_pct"].mean(),
            "p05": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.05),
            "p25": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.25),
            "p75": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.75),
            "p95": rec_df_h["predicted_regret_vs_operator_pct"].quantile(0.95),
        },
        {
            "horizon": h,
            "horizon_name": f"near_{h}",
            "metric": "current_prediction_error_vs_operator_pct = current_pred / actual_operator_mean_speed_t+1_to_h - 1",
            "median": rec_df_h["current_prediction_error_vs_operator_pct"].median(),
            "mean": rec_df_h["current_prediction_error_vs_operator_pct"].mean(),
            "p05": rec_df_h["current_prediction_error_vs_operator_pct"].quantile(0.05),
            "p25": rec_df_h["current_prediction_error_vs_operator_pct"].quantile(0.25),
            "p75": rec_df_h["current_prediction_error_vs_operator_pct"].quantile(0.75),
            "p95": rec_df_h["current_prediction_error_vs_operator_pct"].quantile(0.95),
        },
        {
            "horizon": h,
            "horizon_name": f"near_{h}",
            "metric": "bias_adjusted_regret_vs_operator_pct = regret_vs_operator - current_prediction_error",
            "median": rec_df_h["bias_adjusted_regret_vs_operator_pct"].median(),
            "mean": rec_df_h["bias_adjusted_regret_vs_operator_pct"].mean(),
            "p05": rec_df_h["bias_adjusted_regret_vs_operator_pct"].quantile(0.05),
            "p25": rec_df_h["bias_adjusted_regret_vs_operator_pct"].quantile(0.25),
            "p75": rec_df_h["bias_adjusted_regret_vs_operator_pct"].quantile(0.75),
            "p95": rec_df_h["bias_adjusted_regret_vs_operator_pct"].quantile(0.95),
        },
    ])
    operator_comparison_frames.append(operator_comparison_h)

    win_rate_h = pd.DataFrame([
        {"horizon": h, "horizon_name": f"near_{h}", "metric": "recommended_win_vs_operator_rate", "value": rec_df_h["recommended_win_vs_operator"].mean()},
        {"horizon": h, "horizon_name": f"near_{h}", "metric": "recommended_win_vs_operator_2pct_rate", "value": rec_df_h["recommended_win_vs_operator_2pct"].mean()},
        {"horizon": h, "horizon_name": f"near_{h}", "metric": "recommended_win_vs_operator_5pct_rate", "value": rec_df_h["recommended_win_vs_operator_5pct"].mean()},
        {"horizon": h, "horizon_name": f"near_{h}", "metric": "same_direction_axis_rate", "value": summary_h["same_direction_axis_rate"]},
        {"horizon": h, "horizon_name": f"near_{h}", "metric": "same_direction_rotation_rate", "value": summary_h["same_direction_rotation_rate"]},
        {"horizon": h, "horizon_name": f"near_{h}", "metric": "same_direction_both_rate", "value": summary_h["same_direction_both_rate"]},
    ])
    win_rate_frames.append(win_rate_h)

    all_recommendations.append(rec_df_h)

rec_df_all = pd.concat(all_recommendations, ignore_index=True)
optimizer_summary = pd.DataFrame(optimizer_summary_rows)
by_energy = pd.concat(by_energy_frames, ignore_index=True)
operator_comparison_summary = pd.concat(operator_comparison_frames, ignore_index=True)
win_rate_summary = pd.concat(win_rate_frames, ignore_index=True)

display(optimizer_summary)
display(by_energy)
display(operator_comparison_summary)
display(win_rate_summary)

# Requested compact summary.
requested_metric_summary = optimizer_summary[
    [
        "horizon",
        "horizon_name",
        "median_model_based_predicted_uplift_pct",
        "mean_model_based_predicted_uplift_pct",
        "median_predicted_regret_vs_operator_pct",
        "mean_predicted_regret_vs_operator_pct",
        "recommended_win_vs_operator_rate",
        "recommended_win_vs_operator_2pct_rate",
        "recommended_win_vs_operator_5pct_rate",
        "median_current_prediction_error_vs_operator_pct",
        "median_bias_adjusted_regret_vs_operator_pct",
        "any_boundary_ratio",
    ]
].copy()

display(requested_metric_summary)


Eval rows: 2500
Candidate grid size per row: 441
All candidates: (1102500, 39)
Fast optimizer replay for near_1
Fast optimizer replay for near_3
Fast optimizer replay for near_5


,horizon,horizon_name,optimizer_mode,rows,median_model_based_predicted_uplift_pct,mean_model_based_predicted_uplift_pct,p05_model_based_predicted_uplift_pct,p95_model_based_predicted_uplift_pct,median_predicted_regret_vs_operator_pct,mean_predicted_regret_vs_operator_pct,...,same_direction_both_rate,median_delta_axis_pct,median_delta_rot_pct,median_abs_delta_axis_pct,median_abs_delta_rot_pct,median_operator_delta_axis_pct,median_operator_delta_rot_pct,axis_boundary_ratio,rot_boundary_ratio,any_boundary_ratio
0,1,near_1,light_penalty,2500,1.427325,3.682829,-0.000018,13.257012,4.289630,19.075929,...,0.222403,-5.762057e-12,8.000000e-01,6.439294e-12,8.000000e-01,0.009945,-0.127718,0.0,0.0,0.0
1,3,near_3,light_penalty,2500,-0.000006,1.188091,-0.000017,6.593001,2.781061,6.604164,...,0.114720,-5.595524e-12,-6.539214e-12,6.361578e-12,8.343326e-12,0.021015,0.386586,0.0,0.0,0.0
2,5,near_5,light_penalty,2500,-0.000007,0.668891,-0.000017,3.934041,3.687609,7.152988,...,0.086758,-5.556666e-12,-6.639134e-12,6.428191e-12,7.638334e-12,0.032422,0.648061,0.0,0.0,0.0


,horizon,horizon_name,rock_energy_type_final,rows,median_model_based_predicted_uplift_pct,mean_model_based_predicted_uplift_pct,median_predicted_regret_vs_operator_pct,mean_predicted_regret_vs_operator_pct,recommended_win_vs_operator_rate,recommended_win_vs_operator_2pct_rate,recommended_win_vs_operator_5pct_rate,median_current_prediction_error_vs_operator_pct,median_bias_adjusted_regret_vs_operator_pct,median_abs_delta_axis_pct,median_abs_delta_rot_pct,boundary_ratio
0,1,near_1,hard_high_energy,693,4.687966,5.940083,10.749067,22.022206,0.574315,0.559885,0.542569,5.212981,4.722521,5.773160e-12,2.400000e+00,0.0
1,1,near_1,medium_high_energy,590,4.244938,5.116905,4.942543,19.682090,0.562712,0.528814,0.500000,-0.419531,4.159095,6.239453e-12,3.200000e+00,0.0
2,1,near_1,medium_low_energy,647,1.105468,2.436883,4.107163,16.827146,0.551777,0.528594,0.483771,2.215680,1.063093,6.972201e-12,8.000000e-01,0.0
3,1,near_1,soft_low_energy,570,-0.000005,0.868347,0.170500,17.419010,0.501754,0.482456,0.440351,-0.473224,0.000000,7.377432e-12,9.170442e-12,0.0
4,3,near_3,hard_high_energy,693,-0.000008,2.007347,4.524693,8.055054,0.581530,0.548341,0.493506,3.350421,0.000000,5.651035e-12,1.427747e-11,0.0
5,3,near_3,medium_high_energy,590,-0.000007,1.409062,2.939715,6.563432,0.545763,0.510169,0.452542,1.318705,0.000000,6.089573e-12,8.604228e-12,0.0
6,3,near_3,medium_low_energy,647,-0.000006,0.761936,1.776924,6.100423,0.534776,0.491499,0.429675,0.909461,0.000000,6.861178e-12,7.660539e-12,0.0
7,3,near_3,soft_low_energy,570,-0.000005,0.447045,1.555193,5.454139,0.528070,0.492982,0.433333,1.302174,0.000000,7.438494e-12,8.032464e-12,0.0
8,5,near_5,hard_high_energy,693,-0.000009,1.219886,6.464647,9.292355,0.626263,0.587302,0.525253,5.472008,0.000000,5.684342e-12,8.126833e-12,0.0
9,5,near_5,medium_high_energy,590,-0.000008,0.501684,3.582594,7.060460,0.579661,0.532203,0.477966,2.649235,0.000000,6.100676e-12,7.183143e-12,0.0


,horizon,horizon_name,metric,median,mean,p05,p25,p75,p95
0,1,near_1,model_based_predicted_uplift_pct = recommended...,1.427325,3.682829,-0.000018,-0.000006,6.302944,13.257012
1,1,near_1,predicted_regret_vs_operator_pct = recommended...,4.289630,19.075929,-34.845040,-15.479244,38.283894,113.002228
2,1,near_1,current_prediction_error_vs_operator_pct = cur...,1.342570,14.916419,-37.621557,-18.623131,32.926665,105.912726
3,1,near_1,bias_adjusted_regret_vs_operator_pct = regret_...,1.442598,4.159510,0.000000,0.000000,6.280896,16.757315
4,3,near_3,model_based_predicted_uplift_pct = recommended...,-0.000006,1.188091,-0.000017,-0.000009,1.160747,6.593001
5,3,near_3,predicted_regret_vs_operator_pct = recommended...,2.781061,6.604164,-27.116360,-10.562240,18.445602,54.243218
6,3,near_3,current_prediction_error_vs_operator_pct = cur...,1.729028,5.370927,-28.488707,-11.387317,16.789484,52.109032
7,3,near_3,bias_adjusted_regret_vs_operator_pct = regret_...,0.000000,1.233237,0.000000,0.000000,1.192352,6.635706
8,5,near_5,model_based_predicted_uplift_pct = recommended...,-0.000007,0.668891,-0.000017,-0.000010,0.364839,3.934041
9,5,near_5,predicted_regret_vs_operator_pct = recommended...,3.687609,7.152988,-26.194442,-8.163956,18.295006,50.269348


,horizon,horizon_name,metric,value
0,1,near_1,recommended_win_vs_operator_rate,0.549200
1,1,near_1,recommended_win_vs_operator_2pct_rate,0.526800
2,1,near_1,recommended_win_vs_operator_5pct_rate,0.494000
3,1,near_1,same_direction_axis_rate,0.028269
4,1,near_1,same_direction_rotation_rate,0.300204
5,1,near_1,same_direction_both_rate,0.222403
6,3,near_3,recommended_win_vs_operator_rate,0.548800
7,3,near_3,recommended_win_vs_operator_2pct_rate,0.512000
8,3,near_3,recommended_win_vs_operator_5pct_rate,0.453600
9,3,near_3,same_direction_axis_rate,0.052545


,horizon,horizon_name,median_model_based_predicted_uplift_pct,mean_model_based_predicted_uplift_pct,median_predicted_regret_vs_operator_pct,mean_predicted_regret_vs_operator_pct,recommended_win_vs_operator_rate,recommended_win_vs_operator_2pct_rate,recommended_win_vs_operator_5pct_rate,median_current_prediction_error_vs_operator_pct,median_bias_adjusted_regret_vs_operator_pct,any_boundary_ratio
0,1,near_1,1.427325,3.682829,4.289630,19.075929,0.5492,0.5268,0.4940,1.342570,1.442598,0.0
1,3,near_3,-0.000006,1.188091,2.781061,6.604164,0.5488,0.5120,0.4536,1.729028,0.000000,0.0
2,5,near_5,-0.000007,0.668891,3.687609,7.152988,0.5832,0.5396,0.4736,3.152384,0.000000,0.0


## 11. Well-level drilling time report for default near_5

Этот блок строит таблицу уровня скважины **только по тестовой выборке**:

- одна строка = одна скважина;
- доля каждого класса энергоёмкости внутри скважины;
- фактическое время обуривания по оператору;
- расчётное время при использовании рекомендаций модели;
- экономия времени в минутах и процентах.

Расчёт времени модели является offline counterfactual approximation:

```text
operator_time = sum(dt)
model_time    = sum(interval_depth_m / recommended_predicted_speed_near5)
```

где `interval_depth_m = speed * dt`.

То есть модельное время показывает, сколько заняла бы та же фактическая проходка по глубине, если бы на каждом интервале достигалась скорость, прогнозируемая моделью для выбранного лучшего кандидата. Это не является измеренным причинным эффектом, потому что рекомендации не применялись на реальном станке.


In [12]:
WELL_REPORT_HORIZON = DEFAULT_HORIZON
WELL_REPORT_SOURCE = "test_df"  # fixed: well-level report is calculated only on the test split
WELL_REPORT_CHUNK_ROWS = 2000
WELL_REPORT_MAX_ROWS = None  # set e.g. 20000 for quick debugging

ENERGY_CLASS_ORDER = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

print("Building well-level time report")
print("Horizon:", WELL_REPORT_HORIZON)
print("Source:", WELL_REPORT_SOURCE, "(test split only)")

# Use only the test split to avoid optimistic reporting.
well_source_df = test_df.copy()

well_source_df = well_source_df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

# Real interval time is recomputed here, because feature `dt` may have been filled for first rows.
well_source_df["time_step_sec"] = (
    well_source_df.groupby("well_id")["processing_time"].diff().dt.total_seconds()
)

well_source_df["interval_depth_m"] = well_source_df["speed"] * well_source_df["time_step_sec"]

# Keep intervals that have a physically meaningful time and depth.
well_source_df = well_source_df.replace([np.inf, -np.inf], np.nan)
well_source_df = well_source_df.dropna(
    subset=[
        "time_step_sec",
        "interval_depth_m",
        "speed",
        "pressure_axis",
        "pressure_rotation",
        "rotation",
        "rock_energy_type_final",
    ]
).copy()

well_source_df = well_source_df[
    (well_source_df["time_step_sec"] > 0)
    & (well_source_df["interval_depth_m"] > 0)
    & (well_source_df["speed"] > 0)
].copy()

if WELL_REPORT_MAX_ROWS is not None:
    well_source_df = well_source_df.head(WELL_REPORT_MAX_ROWS).copy()

well_source_df = well_source_df.reset_index(drop=True)
well_source_df["row_id"] = np.arange(len(well_source_df))

print("Rows for well report:", len(well_source_df))
print("Wells for well report:", well_source_df["well_id"].nunique())


def build_candidate_grid_batch(rows: pd.DataFrame, grid_size: int = GRID_SIZE) -> pd.DataFrame:
    rows = rows.reset_index(drop=True).copy()
    n = len(rows)
    if n == 0:
        return pd.DataFrame()

    fractions = np.linspace(0.0, 1.0, grid_size)
    fa, fr = np.meshgrid(fractions, fractions)
    fa = fa.ravel()
    fr = fr.ravel()
    k = len(fa)

    row_rep = rows.loc[rows.index.repeat(k)].reset_index(drop=True)

    cur_ax = rows["pressure_axis"].to_numpy(dtype=float)
    cur_rotp = rows["pressure_rotation"].to_numpy(dtype=float)

    # Energy-type global bounds.
    default_ax_low = float(train_df["pressure_axis"].quantile(0.05))
    default_ax_high = float(train_df["pressure_axis"].quantile(0.95))
    default_rot_low = float(train_df["pressure_rotation"].quantile(0.05))
    default_rot_high = float(train_df["pressure_rotation"].quantile(0.95))

    ax_low_global = np.array([
        surface_ranges.get(et, {}).get("pressure_axis_q05", default_ax_low)
        for et in rows["rock_energy_type_final"]
    ], dtype=float)
    ax_high_global = np.array([
        surface_ranges.get(et, {}).get("pressure_axis_q95", default_ax_high)
        for et in rows["rock_energy_type_final"]
    ], dtype=float)
    rot_low_global = np.array([
        surface_ranges.get(et, {}).get("pressure_rotation_q05", default_rot_low)
        for et in rows["rock_energy_type_final"]
    ], dtype=float)
    rot_high_global = np.array([
        surface_ranges.get(et, {}).get("pressure_rotation_q95", default_rot_high)
        for et in rows["rock_energy_type_final"]
    ], dtype=float)

    local_ax_low = cur_ax * (1.0 - MAX_DELTA_FRAC)
    local_ax_high = cur_ax * (1.0 + MAX_DELTA_FRAC)
    local_rot_low = cur_rotp * (1.0 - MAX_DELTA_FRAC)
    local_rot_high = cur_rotp * (1.0 + MAX_DELTA_FRAC)

    ax_min = np.maximum(ax_low_global, local_ax_low)
    ax_max = np.minimum(ax_high_global, local_ax_high)
    rot_min = np.maximum(rot_low_global, local_rot_low)
    rot_max = np.minimum(rot_high_global, local_rot_high)

    # If global quantile bounds collapse local range, fall back to local bounds.
    bad_ax = ax_min >= ax_max
    ax_min[bad_ax] = local_ax_low[bad_ax]
    ax_max[bad_ax] = local_ax_high[bad_ax]

    bad_rot = rot_min >= rot_max
    rot_min[bad_rot] = local_rot_low[bad_rot]
    rot_max[bad_rot] = local_rot_high[bad_rot]

    ax_values = np.repeat(ax_min, k) + np.tile(fa, n) * np.repeat((ax_max - ax_min), k)
    rot_values = np.repeat(rot_min, k) + np.tile(fr, n) * np.repeat((rot_max - rot_min), k)

    cand = row_rep[base_numeric_features + categorical_features + ["row_id"]].copy()
    cand["pressure_axis"] = ax_values
    cand["pressure_rotation"] = rot_values

    cand = recompute_candidate_features(cand)

    original_axis = row_rep["pressure_axis"].to_numpy(dtype=float)
    original_rot = row_rep["pressure_rotation"].to_numpy(dtype=float)

    cand["delta_pressure_axis_frac"] = cand["pressure_axis"].to_numpy(dtype=float) / (original_axis + EPS) - 1.0
    cand["delta_pressure_rotation_frac"] = cand["pressure_rotation"].to_numpy(dtype=float) / (original_rot + EPS) - 1.0

    return drop_duplicate_columns(cand)


def recommend_best_for_rows_chunked(rows: pd.DataFrame, horizon: int) -> pd.DataFrame:
    rotation_model = models_by_horizon[horizon]["rotation_model"]
    speed_model = models_by_horizon[horizon]["speed_model"]

    base_feature_cols = base_numeric_features + categorical_features
    speed_feature_cols = speed_numeric_features + categorical_features

    best_chunks = []

    for start in range(0, len(rows), WELL_REPORT_CHUNK_ROWS):
        chunk = rows.iloc[start:start + WELL_REPORT_CHUNK_ROWS].copy()

        cand = build_candidate_grid_batch(chunk, grid_size=GRID_SIZE)
        cand = drop_duplicate_columns(cand)

        cand["candidate_target_rotation"] = rotation_model.predict(cand[base_feature_cols])
        cand["predicted_target_speed"] = speed_model.predict(cand[speed_feature_cols])
        cand = score_candidates(cand)

        best_idx = cand.groupby("row_id", sort=False)["optimizer_score"].idxmax()
        best = cand.loc[best_idx, [
            "row_id",
            "pressure_axis",
            "pressure_rotation",
            "candidate_target_rotation",
            "predicted_target_speed",
            "optimizer_score",
            "delta_pressure_axis_frac",
            "delta_pressure_rotation_frac",
        ]].copy()

        best_chunks.append(best)

        if (start // WELL_REPORT_CHUNK_ROWS) % 10 == 0:
            print(f"processed rows: {min(start + WELL_REPORT_CHUNK_ROWS, len(rows))} / {len(rows)}")

    return pd.concat(best_chunks, ignore_index=True)


best_recs = recommend_best_for_rows_chunked(well_source_df, horizon=WELL_REPORT_HORIZON)

row_level_time = well_source_df.merge(
    best_recs,
    on="row_id",
    how="left",
    suffixes=("", "_recommended"),
)

row_level_time = row_level_time.rename(columns={
    "pressure_axis_recommended": "recommended_pressure_axis",
    "pressure_rotation_recommended": "recommended_pressure_rotation",
    "candidate_target_rotation": "recommended_candidate_target_rotation",
    "predicted_target_speed": "recommended_predicted_speed",
})

row_level_time["model_time_step_sec"] = (
    row_level_time["interval_depth_m"]
    / (row_level_time["recommended_predicted_speed"] + EPS)
)

row_level_time["operator_time_step_sec"] = row_level_time["time_step_sec"]

row_level_time = row_level_time.replace([np.inf, -np.inf], np.nan)
row_level_time = row_level_time.dropna(
    subset=["operator_time_step_sec", "model_time_step_sec", "interval_depth_m"]
).copy()

# Rock-energy shares by depth. If interval depth is available, this is better than row-count share.
energy_depth = (
    row_level_time
    .groupby(["well_id", "rock_energy_type_final"])["interval_depth_m"]
    .sum()
    .reset_index()
)

energy_pivot = energy_depth.pivot_table(
    index="well_id",
    columns="rock_energy_type_final",
    values="interval_depth_m",
    aggfunc="sum",
    fill_value=0.0,
)

energy_pivot["energy_total_depth_m"] = energy_pivot.sum(axis=1)

# Ensure four expected columns exist.
for cls in ENERGY_CLASS_ORDER:
    if cls not in energy_pivot.columns:
        energy_pivot[cls] = 0.0

for cls in ENERGY_CLASS_ORDER:
    energy_pivot[f"{cls}_pct"] = 100.0 * (
        energy_pivot[cls] / (energy_pivot["energy_total_depth_m"] + EPS)
    )

energy_share_cols = [f"{cls}_pct" for cls in ENERGY_CLASS_ORDER]

# Optional unknown/other share if present.
known_depth = energy_pivot[ENERGY_CLASS_ORDER].sum(axis=1)
energy_pivot["other_or_unknown_energy_pct"] = 100.0 * (
    (energy_pivot["energy_total_depth_m"] - known_depth)
    / (energy_pivot["energy_total_depth_m"] + EPS)
)

energy_pivot = energy_pivot.reset_index()[["well_id"] + energy_share_cols + ["other_or_unknown_energy_pct"]]

well_time_report = (
    row_level_time
    .groupby("well_id")
    .agg(
        telemetry_rows_used=("row_id", "size"),
        drilling_start=("processing_time", "min"),
        drilling_end=("processing_time", "max"),
        total_depth_modeled_m=("interval_depth_m", "sum"),
        operator_time_sec=("operator_time_step_sec", "sum"),
        model_time_sec=("model_time_step_sec", "sum"),
        current_speed_mean=("speed", "mean"),
        recommended_predicted_speed_mean=("recommended_predicted_speed", "mean"),
        current_pressure_axis_mean=("pressure_axis", "mean"),
        current_pressure_rotation_mean=("pressure_rotation", "mean"),
        recommended_pressure_axis_mean=("recommended_pressure_axis", "mean"),
        recommended_pressure_rotation_mean=("recommended_pressure_rotation", "mean"),
    )
    .reset_index()
)

well_time_report["operator_time_min"] = well_time_report["operator_time_sec"] / 60.0
well_time_report["model_time_min"] = well_time_report["model_time_sec"] / 60.0
well_time_report["time_saved_sec"] = well_time_report["operator_time_sec"] - well_time_report["model_time_sec"]
well_time_report["time_saved_min"] = well_time_report["time_saved_sec"] / 60.0
well_time_report["time_saved_pct"] = 100.0 * (
    well_time_report["time_saved_sec"]
    / (well_time_report["operator_time_sec"] + EPS)
)
well_time_report["model_time_vs_operator_pct"] = 100.0 * (
    well_time_report["model_time_sec"]
    / (well_time_report["operator_time_sec"] + EPS)
    - 1.0
)

well_time_report = well_time_report.merge(energy_pivot, on="well_id", how="left")
well_time_report[energy_share_cols + ["other_or_unknown_energy_pct"]] = (
    well_time_report[energy_share_cols + ["other_or_unknown_energy_pct"]].fillna(0.0)
)

# Friendly column order.
well_time_report = well_time_report[
    [
        "well_id",
        "telemetry_rows_used",
        "drilling_start",
        "drilling_end",
        "total_depth_modeled_m",
        *energy_share_cols,
        "other_or_unknown_energy_pct",
        "operator_time_min",
        "model_time_min",
        "time_saved_min",
        "time_saved_pct",
        "model_time_vs_operator_pct",
        "current_speed_mean",
        "recommended_predicted_speed_mean",
        "current_pressure_axis_mean",
        "recommended_pressure_axis_mean",
        "current_pressure_rotation_mean",
        "recommended_pressure_rotation_mean",
    ]
].sort_values("well_id").reset_index(drop=True)

well_time_summary = pd.DataFrame([
    {
        "horizon": WELL_REPORT_HORIZON,
        "source": WELL_REPORT_SOURCE,
        "wells": int(len(well_time_report)),
        "rows_used": int(len(row_level_time)),
        "operator_time_total_min": float(well_time_report["operator_time_min"].sum()),
        "model_time_total_min": float(well_time_report["model_time_min"].sum()),
        "time_saved_total_min": float(well_time_report["time_saved_min"].sum()),
        "time_saved_total_pct": float(
            100.0
            * well_time_report["time_saved_min"].sum()
            / (well_time_report["operator_time_min"].sum() + EPS)
        ),
        "median_well_time_saved_pct": float(well_time_report["time_saved_pct"].median()),
        "mean_well_time_saved_pct": float(well_time_report["time_saved_pct"].mean()),
        "p05_well_time_saved_pct": float(well_time_report["time_saved_pct"].quantile(0.05)),
        "p95_well_time_saved_pct": float(well_time_report["time_saved_pct"].quantile(0.95)),
    }
])

display(well_time_summary)
display(well_time_report.head(20))
display(well_time_report.describe(percentiles=[.05, .25, .5, .75, .95]))

# Save directly, because this is the table intended for report work.
well_time_report.to_csv(REPORT_DIR / "well_level_time_report_near5.csv", index=False)
well_time_summary.to_csv(REPORT_DIR / "well_level_time_summary_near5.csv", index=False)

# Also save to artifacts because it may be useful downstream, but it is one compact file.
well_time_report.to_csv(ARTIFACT_DIR / "well_level_time_report_near5.csv", index=False)
well_time_summary.to_csv(ARTIFACT_DIR / "well_level_time_summary_near5.csv", index=False)

print("Saved:")
print(REPORT_DIR / "well_level_time_report_near5.csv")
print(REPORT_DIR / "well_level_time_summary_near5.csv")


Building well-level time report
Horizon: 5
Source: test_df (test split only)
Rows for well report: 92931
Wells for well report: 427
processed rows: 2000 / 92931
processed rows: 22000 / 92931
processed rows: 42000 / 92931
processed rows: 62000 / 92931
processed rows: 82000 / 92931


,horizon,source,wells,rows_used,operator_time_total_min,model_time_total_min,time_saved_total_min,time_saved_total_pct,median_well_time_saved_pct,mean_well_time_saved_pct,p05_well_time_saved_pct,p95_well_time_saved_pct
0,5,test_df,427,92931,10519.281583,10007.718253,511.56333,4.863101,4.199934,4.720077,-6.653451,18.971953


,well_id,telemetry_rows_used,drilling_start,drilling_end,total_depth_modeled_m,soft_low_energy_pct,medium_low_energy_pct,medium_high_energy_pct,hard_high_energy_pct,other_or_unknown_energy_pct,...,model_time_min,time_saved_min,time_saved_pct,model_time_vs_operator_pct,current_speed_mean,recommended_predicted_speed_mean,current_pressure_axis_mean,recommended_pressure_axis_mean,current_pressure_rotation_mean,recommended_pressure_rotation_mean
0,19677,192,2025-08-25 00:12:51.173,2025-08-25 00:29:27.519,17.305746,100.000000,0.000000,0.000000,0.000000,0.000000e+00,...,16.672496,0.010138,0.060767,-0.060767,0.017296,0.017026,17147.364583,17203.995875,14509.156250,14505.860927
1,19720,327,2025-08-25 06:06:42.284,2025-08-25 06:39:06.366,17.270179,0.000000,0.000000,14.938499,85.061501,0.000000e+00,...,31.261684,1.223783,3.767170,-3.767170,0.009110,0.009293,18833.880734,18832.035376,14125.042813,14173.930235
2,19775,255,2025-08-25 20:59:24.250,2025-08-25 21:32:24.971,20.201009,43.358793,17.565288,5.360487,33.715432,0.000000e+00,...,28.309683,4.778850,14.442617,-14.442617,0.011132,0.011224,15536.188235,15562.055224,13316.909804,13333.430353
3,19780,208,2025-08-25 21:41:26.288,2025-08-25 22:06:37.467,30.117016,60.957494,39.042506,0.000000,0.000000,0.000000e+00,...,29.165613,-3.910913,-15.485882,15.485882,0.014408,0.013739,14865.115385,14894.000865,14065.105769,14045.414904
4,19789,234,2025-08-25 22:55:55.460,2025-08-25 23:26:03.522,19.459912,58.210726,18.762721,23.026553,0.000000,0.000000e+00,...,24.622605,5.604328,18.540844,-18.540844,0.013031,0.013417,16184.965812,16212.058701,14049.961538,14080.052957
5,19905,101,2025-08-26 17:39:45.077,2025-08-26 18:07:10.524,30.467900,0.000000,0.000000,0.000000,100.000000,0.000000e+00,...,33.818215,-6.305631,-22.919082,22.919082,0.009896,0.009934,12980.178218,13000.386099,12985.990099,12985.848941
6,19909,280,2025-08-26 19:49:13.054,2025-08-26 20:18:32.433,20.535202,70.711436,0.000000,29.288564,0.000000,0.000000e+00,...,25.866646,3.519804,11.977643,-11.977643,0.012680,0.013100,15979.314286,15995.667714,13470.385714,13497.909186
7,19935,226,2025-08-26 22:31:52.344,2025-08-26 22:51:33.426,15.503191,16.912229,53.611204,29.476567,0.000000,0.000000e+00,...,18.879102,0.891281,4.508164,-4.508164,0.013348,0.013517,19259.190265,19244.176310,15327.349558,15306.792602
8,19958,256,2025-08-27 00:09:00.283,2025-08-27 00:31:50.078,17.274083,0.000000,40.176274,59.823726,0.000000,0.000000e+00,...,22.196789,0.638794,2.797364,-2.797364,0.012766,0.012858,21131.765625,20473.147187,14974.523438,15013.932648
9,19977,220,2025-08-27 02:37:34.001,2025-08-27 02:57:02.061,18.167067,26.097171,46.620566,27.282263,0.000000,0.000000e+00,...,20.028341,-0.469774,-2.401885,2.401885,0.015299,0.014885,19339.045455,19045.530082,15441.186364,15424.857118


,well_id,telemetry_rows_used,drilling_start,drilling_end,total_depth_modeled_m,soft_low_energy_pct,medium_low_energy_pct,medium_high_energy_pct,hard_high_energy_pct,other_or_unknown_energy_pct,...,model_time_min,time_saved_min,time_saved_pct,model_time_vs_operator_pct,current_speed_mean,recommended_predicted_speed_mean,current_pressure_axis_mean,recommended_pressure_axis_mean,current_pressure_rotation_mean,recommended_pressure_rotation_mean
count,427.000000,427.000000,427,427,427.000000,427.000000,427.000000,427.000000,427.000000,4.270000e+02,...,427.000000,427.000000,427.000000,427.000000,427.000000,427.000000,427.000000,427.000000,427.000000,427.000000
mean,24781.039813,217.637002,2025-10-01 03:42:05.032597,2025-10-01 04:06:36.646374,19.074861,34.306731,25.821880,21.037311,18.834078,2.843253e-16,...,23.437279,1.198041,4.720077,-4.720077,0.014369,0.014415,18254.878303,18234.387425,14645.015485,14647.643485
min,19677.000000,19.000000,2025-08-25 00:12:51.173000,2025-08-25 00:29:27.519000,2.053083,0.000000,0.000000,0.000000,0.000000,-2.167953e-14,...,2.306543,-108.477967,-85.165166,-65.794273,0.006150,0.007106,7426.765957,7432.788085,8605.398936,8694.492989
5%,20234.000000,74.300000,2025-08-29 01:37:00.238100,2025-08-29 02:13:22.551800,6.939012,0.000000,0.000000,0.000000,0.000000,0.000000e+00,...,7.822455,-0.849852,-6.653451,-18.971953,0.008396,0.008952,13896.225831,14069.015127,11594.410034,11614.699250
25%,22132.000000,137.000000,2025-09-13 20:33:15.614000,2025-09-13 20:51:46.045000,13.501208,0.000000,0.000000,0.000000,0.000000,0.000000e+00,...,13.445126,0.032835,0.235023,-9.095251,0.011019,0.011318,16522.519038,16564.107956,13413.494017,13417.866293
50%,24010.000000,208.000000,2025-09-26 19:52:07.291000,2025-09-26 20:13:35.294000,16.192040,18.819910,16.661348,6.203058,0.000000,0.000000e+00,...,20.556278,0.763755,4.199934,-4.199934,0.014072,0.014147,18603.965409,18601.179596,14789.812155,14797.198760
75%,27608.000000,286.000000,2025-10-21 14:20:02.511000,2025-10-21 14:30:52.947500,20.767698,65.711607,44.626048,36.533219,33.557947,0.000000e+00,...,28.453137,2.518748,9.095251,-0.235023,0.017079,0.016968,20086.622928,20072.450558,15957.682676,15986.349511
95%,30925.300000,398.800000,2025-11-07 13:12:06.373800,2025-11-07 15:14:35.502400,29.497855,100.000000,88.883720,80.529235,87.647321,0.000000e+00,...,41.525887,7.072826,18.971953,6.653451,0.022238,0.021463,21444.169515,21399.189667,17196.643249,17155.591327
max,31923.000000,584.000000,2025-11-11 12:51:58.981000,2025-11-11 13:10:22.179000,246.427924,100.000000,100.000000,100.000000,100.000000,2.544215e-14,...,235.851600,73.453240,65.794273,85.165166,0.032449,0.029725,23230.072581,22747.668516,19693.644444,19462.829433
std,3351.666721,102.956173,NaN,NaN,18.414333,38.932291,29.697233,27.792973,30.588003,4.160260e-15,...,21.072601,9.542987,10.684949,10.684949,0.004278,0.003890,2521.518234,2467.455431,1824.195491,1795.738640


Saved:
C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\drilling_advisory_light_penalty_reports\well_level_time_report_near5.csv
C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\drilling_advisory_light_penalty_reports\well_level_time_summary_near5.csv


## 12. Example recommendation for default near_5

In [13]:
example_row = test_df.sample(1, random_state=RANDOM_STATE + 10).iloc[0]
example_h = DEFAULT_HORIZON
rec = recommend_for_row(example_row, horizon=example_h, grid_size=31)

target_speed = target_cols_by_horizon[example_h]["target_speed"]

example_actual_target_speed = float(example_row[target_speed])
example_current_pred_speed = float(rec["current_predicted_target_speed"])
example_recommended_pred_speed = float(rec["predicted_target_speed"])

example_predicted_regret_vs_operator_pct = 100.0 * (
    example_recommended_pred_speed / (example_actual_target_speed + EPS) - 1.0
)
example_current_error_vs_operator_pct = 100.0 * (
    example_current_pred_speed / (example_actual_target_speed + EPS) - 1.0
)
example_bias_adjusted_regret_pct = (
    example_predicted_regret_vs_operator_pct - example_current_error_vs_operator_pct
)

example_summary = pd.DataFrame([
    {
        "horizon": f"near_{example_h}",
        "variant": "operator/current",
        "pressure_axis": example_row["pressure_axis"],
        "pressure_rotation": example_row["pressure_rotation"],
        "current_speed": example_row["speed"],
        "target_speed_actual": example_actual_target_speed,
        "predicted_target_speed": example_current_pred_speed,
        "model_based_predicted_uplift_pct": 0.0,
        "predicted_regret_vs_operator_pct": example_current_error_vs_operator_pct,
        "current_prediction_error_vs_operator_pct": example_current_error_vs_operator_pct,
        "bias_adjusted_regret_vs_operator_pct": 0.0,
        "delta_axis_pct": 0.0,
        "delta_rot_pct": 0.0,
    },
    {
        "horizon": f"near_{example_h}",
        "variant": FINAL_OPTIMIZER_MODE,
        "pressure_axis": rec["recommended_pressure_axis"],
        "pressure_rotation": rec["recommended_pressure_rotation"],
        "current_speed": example_row["speed"],
        "target_speed_actual": example_actual_target_speed,
        "predicted_target_speed": example_recommended_pred_speed,
        "optimizer_score": rec["optimizer_score"],
        "model_based_predicted_uplift_pct": rec["predicted_uplift_pct"],
        "score_uplift_pct": rec["score_uplift_pct"],
        "predicted_regret_vs_operator_pct": example_predicted_regret_vs_operator_pct,
        "current_prediction_error_vs_operator_pct": example_current_error_vs_operator_pct,
        "bias_adjusted_regret_vs_operator_pct": example_bias_adjusted_regret_pct,
        "delta_axis_pct": rec["delta_pressure_axis_pct"],
        "delta_rot_pct": rec["delta_pressure_rotation_pct"],
    },
])

print("Example well:", example_row["well_id"])
print("Example time:", example_row["processing_time"])
print("Energy type:", example_row["rock_energy_type_final"])
display(example_summary)


Example well: 24421
Example time: 2025-09-30 05:15:50.592000
Energy type: hard_high_energy


,horizon,variant,pressure_axis,pressure_rotation,current_speed,target_speed_actual,predicted_target_speed,model_based_predicted_uplift_pct,predicted_regret_vs_operator_pct,current_prediction_error_vs_operator_pct,bias_adjusted_regret_vs_operator_pct,delta_axis_pct,delta_rot_pct,optimizer_score,score_uplift_pct
0,near_5,operator/current,12933.0,9916.000000,0.00606,0.006207,0.009430,0.000000,51.922987,51.922987,0.000000,0.000000,0.000000,NaN,NaN
1,near_5,light_penalty,13524.0,9922.142667,0.00606,0.006207,0.009228,-2.141551,48.669494,51.922987,-3.253493,4.569705,0.061947,0.008765,-7.053309


## 13. Save compact runtime artifacts and reports

In [14]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save only artifacts used by simulator/runtime + compact reports used for analysis.
# Removed noisy files:
# - offline_recommendations_near1/near3/near5_*.csv
# - offline_recommendations_light_penalty_all_horizons.csv
# - repeated report copies of every table
# - uplift_compare.csv duplicate
#
# Kept runtime artifacts:
# - models for near_1/near_3/near_5
# - default near_5 model names for backward compatibility
# - feature_config.json
# - optimizer_config.json
# - surface_ranges_by_energy_type.json
#
# Kept compact analysis reports:
# - training_metrics_by_horizon.csv
# - requested_metric_summary_by_horizon.csv
# - optimizer_summary_by_horizon.csv
# - operator_comparison_summary_by_horizon.csv
# - win_rate_summary_by_horizon.csv
# - uplift_by_energy_type_by_horizon.csv

for h, result in models_by_horizon.items():
    joblib.dump(result["rotation_model"], ARTIFACT_DIR / f"rotation_model_near{h}.joblib")
    joblib.dump(result["speed_model"], ARTIFACT_DIR / f"speed_model_near{h}.joblib")

# Backward-compatible names for simulator/default horizon.
joblib.dump(models_by_horizon[DEFAULT_HORIZON]["rotation_model"], ARTIFACT_DIR / "rotation_model_near5.joblib")
joblib.dump(models_by_horizon[DEFAULT_HORIZON]["speed_model"], ARTIFACT_DIR / "speed_model_near5.joblib")

with open(ARTIFACT_DIR / "surface_ranges_by_energy_type.json", "w", encoding="utf-8") as f:
    json.dump(surface_ranges, f, ensure_ascii=False, indent=2)

feature_config = {
    "data_path": str(DATA_PATH),
    "rock_energy_segmentation_expected_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "target_horizons": TARGET_HORIZONS,
    "default_horizon": DEFAULT_HORIZON,
    "target_columns_by_horizon": target_cols_by_horizon,
    "numeric_features": base_numeric_features,
    "base_numeric_features": base_numeric_features,
    "categorical_features": categorical_features,
    "rotation_features": base_numeric_features + categorical_features,
    "speed_features": speed_numeric_features + categorical_features,
    "speed_numeric_features": speed_numeric_features,
    "speed_extra_features": speed_extra_features,
    "removed_pruned_features": PRUNED_FEATURES_REMOVED,
    "feature_pruning_mode": "aggressive_correlation_and_low_importance_pruning",
    "hardness_feature_columns": HARDNESS_FEATURE_COLUMNS,
    "hardness_feature_policy": "single stable log-pseudo-MSE hardness feature; energy class based on 60-window segment quantiles",
    "energy_type_column": "rock_energy_type_final",
    "required_live_columns": [
        "processing_time", "well_id", "pressure_axis", "pressure_rotation",
        "rotation", "speed", "hardness_score_smooth", "rock_energy_type_final",
    ],
}

with open(ARTIFACT_DIR / "feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, ensure_ascii=False, indent=2)

optimizer_config = {
    "optimizer_mode": FINAL_OPTIMIZER_MODE,
    "target_horizons": TARGET_HORIZONS,
    "default_horizon": DEFAULT_HORIZON,
    "grid_size_default": int(GRID_SIZE),
    "max_delta_frac_default": float(MAX_DELTA_FRAC),
    "change_penalty_weight": float(CHANGE_PENALTY_WEIGHT),
    "boundary_penalty_weight": float(BOUNDARY_PENALTY_WEIGHT),
    "boundary_start": float(BOUNDARY_START),
    "score_formula": "pred_target_speed - change_penalty - boundary_penalty",
    "uplift_formula_model_based": "100 * (recommended_predicted_target_speed / current_predicted_target_speed - 1)",
    "predicted_regret_vs_operator_formula": "100 * (recommended_predicted_target_speed / actual_operator_mean_speed_t+1_to_h - 1)",
    "recommended_win_vs_operator_formula": "recommended_predicted_target_speed > actual_operator_mean_speed_t+1_to_h * (1 + margin)",
    "current_prediction_error_formula": "100 * (current_predicted_target_speed / actual_operator_mean_speed_t+1_to_h - 1)",
    "bias_adjusted_regret_formula": "predicted_regret_vs_operator_pct - current_prediction_error_vs_operator_pct",
    "direction_agreement_formula": "sign(recommended_delta_pressure_pct) == sign(operator_future_delta_pressure_pct), with a 0.5 pp deadband",
    "operator_comparison_interpretation": "offline counterfactual diagnostics; not a measured causal effect because the recommendation was not actually applied",
    "use_local_reachable_bounds": True,
    "use_energy_type_quantile_bounds": True,
}

with open(ARTIFACT_DIR / "optimizer_config.json", "w", encoding="utf-8") as f:
    json.dump(optimizer_config, f, ensure_ascii=False, indent=2)

training_report = {
    "model_type": "LightGBM LGBMRegressor",
    "rock_energy_segmentation_expected_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "target_horizons": TARGET_HORIZONS,
    "default_horizon": DEFAULT_HORIZON,
    "metrics_by_horizon": metrics_summary.to_dict(orient="records"),
    "optimizer_summary_by_horizon": optimizer_summary.to_dict(orient="records"),
    "operator_comparison_summary_by_horizon": operator_comparison_summary.to_dict(orient="records"),
    "win_rate_summary_by_horizon": win_rate_summary.to_dict(orient="records"),
    "requested_metric_summary_by_horizon": requested_metric_summary.to_dict(orient="records"),
    "final_optimizer_mode": FINAL_OPTIMIZER_MODE,
    "feature_pruning_mode": "aggressive_correlation_and_low_importance_pruning",
    "removed_pruned_features": PRUNED_FEATURES_REMOVED,
    "hardness_feature_columns": HARDNESS_FEATURE_COLUMNS,
}

with open(ARTIFACT_DIR / "training_report.json", "w", encoding="utf-8") as f:
    json.dump(training_report, f, ensure_ascii=False, indent=2)

# Compact CSVs in artifact dir.
metrics_summary.to_csv(ARTIFACT_DIR / "training_metrics_by_horizon.csv", index=False)
requested_metric_summary.to_csv(ARTIFACT_DIR / "requested_metric_summary_by_horizon.csv", index=False)
optimizer_summary.to_csv(ARTIFACT_DIR / "optimizer_summary_by_horizon.csv", index=False)
operator_comparison_summary.to_csv(ARTIFACT_DIR / "operator_comparison_summary_by_horizon.csv", index=False)
win_rate_summary.to_csv(ARTIFACT_DIR / "win_rate_summary_by_horizon.csv", index=False)
by_energy.to_csv(ARTIFACT_DIR / "uplift_by_energy_type_by_horizon.csv", index=False)

# Backward-compatible default near_5 compact reports.
default_optimizer_summary = optimizer_summary[optimizer_summary["horizon"] == DEFAULT_HORIZON].copy()
default_by_energy = by_energy[by_energy["horizon"] == DEFAULT_HORIZON].copy()
default_operator_summary = operator_comparison_summary[operator_comparison_summary["horizon"] == DEFAULT_HORIZON].copy()
default_win_rate = win_rate_summary[win_rate_summary["horizon"] == DEFAULT_HORIZON].copy()
default_requested_summary = requested_metric_summary[requested_metric_summary["horizon"] == DEFAULT_HORIZON].copy()

default_optimizer_summary.to_csv(ARTIFACT_DIR / "optimizer_summary.csv", index=False)
default_by_energy.to_csv(ARTIFACT_DIR / "uplift_by_energy_type.csv", index=False)
default_operator_summary.to_csv(ARTIFACT_DIR / "operator_comparison_summary.csv", index=False)
default_win_rate.to_csv(ARTIFACT_DIR / "win_rate_summary.csv", index=False)
default_requested_summary.to_csv(ARTIFACT_DIR / "requested_metric_summary.csv", index=False)

# Mirror only compact reports to REPORT_DIR for easy viewing.
metrics_summary.to_csv(REPORT_DIR / "training_metrics_by_horizon.csv", index=False)
requested_metric_summary.to_csv(REPORT_DIR / "requested_metric_summary_by_horizon.csv", index=False)
optimizer_summary.to_csv(REPORT_DIR / "optimizer_summary_by_horizon.csv", index=False)
operator_comparison_summary.to_csv(REPORT_DIR / "operator_comparison_summary_by_horizon.csv", index=False)
win_rate_summary.to_csv(REPORT_DIR / "win_rate_summary_by_horizon.csv", index=False)
by_energy.to_csv(REPORT_DIR / "uplift_by_energy_type_by_horizon.csv", index=False)

print("Saved compact artifacts to:", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", p.name)

print("\nSaved compact reports to:", REPORT_DIR.resolve())
for p in sorted(REPORT_DIR.iterdir()):
    print(" -", p.name)

print("\nRequested summary:")
display(requested_metric_summary)


Saved compact artifacts to: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\drilling_advisory_light_penalty_artifacts
 - baseline_compare_by_horizon.csv
 - feature_config.json
 - offline_recommendations_light_penalty.csv
 - offline_recommendations_light_penalty_all_horizons.csv
 - offline_recommendations_near1_light_penalty.csv
 - offline_recommendations_near3_light_penalty.csv
 - offline_recommendations_near5_light_penalty.csv
 - operator_comparison_summary.csv
 - operator_comparison_summary_by_horizon.csv
 - optimizer_config.json
 - optimizer_summary.csv
 - optimizer_summary_by_horizon.csv
 - requested_metric_summary.csv
 - requested_metric_summary_by_horizon.csv
 - rotation_model_near1.joblib
 - rotation_model_near3.joblib
 - rotation_model_near5.joblib
 - speed_model_near1.joblib
 - speed_model_near3.joblib
 - speed_model_near5.joblib
 - surface_ranges_by_energy_type.json
 - training_metrics.csv
 - training_metrics_by_horizon.csv
 - training_report.json
 - up

,horizon,horizon_name,median_model_based_predicted_uplift_pct,mean_model_based_predicted_uplift_pct,median_predicted_regret_vs_operator_pct,mean_predicted_regret_vs_operator_pct,recommended_win_vs_operator_rate,recommended_win_vs_operator_2pct_rate,recommended_win_vs_operator_5pct_rate,median_current_prediction_error_vs_operator_pct,median_bias_adjusted_regret_vs_operator_pct,any_boundary_ratio
0,1,near_1,1.427325,3.682829,4.289630,19.075929,0.5492,0.5268,0.4940,1.342570,1.442598,0.0
1,3,near_3,-0.000006,1.188091,2.781061,6.604164,0.5488,0.5120,0.4536,1.729028,0.000000,0.0
2,5,near_5,-0.000007,0.668891,3.687609,7.152988,0.5832,0.5396,0.4736,3.152384,0.000000,0.0
